# Imports

In [36]:
import pandas as pd
import numpy as np
import pyarrow
import matplotlib.pyplot as plt
import re
import pandas as pd
import os

In [31]:
data=pd.read_csv("data/raw/buildstock/MN_upgrade0.csv", dtype={'MN_upgrade0.csv': 'str'})

/tmp/ipykernel_7755/105414082.py:1: DtypeWarning: Columns (5,144,151,152,359,360) have mixed types. Specify dtype option on import or set low_memory=False.
  data=pd.read_csv("data/raw/buildstock/MN_upgrade0.csv", dtype={'MN_upgrade0.csv': 'str'})


# Constants
Daddy's made you some constants.

In [32]:
def get_parquet_file_path(state_abbreviation: str, building_id: str) -> str:
    """Returns the directory path for parquet files based on the state abbreviation."""
    return f"data/raw/nrel_parquet/{state_abbreviation.lower()}/{building_id}-0.parquet"

columns_of_interest = [
    "MN_upgrade0.csv",
    "in.city",
    "in.sqft..ft2",
    'in.heating_fuel',
    'in.heating_setpoint',
    "in.insulation_ceiling",
    "in.insulation_floor",
    "in.insulation_foundation_wall",
    "in.insulation_rim_joist",
    "in.insulation_roof",
    "in.geometry_building_type_recs",
    "out.params.size_heat_pump_backup_primary..kbtu_per_hr",
    "out.params.size_heating_system_primary..kbtu_per_hr",
    "out.params.size_heating_system_secondary..kbtu_per_hr"
]

column_renames = {
    "MN_upgrade0.csv": "building_id",
    "in.city": "city",
    "in.sqft..ft2": 'area',
    "in.heating_fuel": 'heating_fuel',
    "in.heating_setpoint": 'heating_setpoint',
    "in.insulation_ceiling": 'ceiling_r',
    "in.insulation_floor": 'floor_r',
    "in.insulation_foundation_wall": 'foundation_wall_r',
    "in.insulation_rim_joist": 'rim_joist_r',
    "in.insulation_roof": 'roof_r',
    "in.geometry_building_type_recs": 'building_type',
    "out.params.size_heat_pump_backup_primary..kbtu_per_hr": 'heat_pump_backup_primary_size',
    "out.params.size_heating_system_primary..kbtu_per_hr": 'heating_system_primary_size',
    "out.params.size_heating_system_secondary..kbtu_per_hr": 'heating_system_secondary_size'
}

df = data[columns_of_interest].copy()
df.rename(columns=column_renames, inplace=True)
df['building_id'] = df['building_id'].astype('str')

# filter file down to only the columns of interest and only the rows for Minneapolis, MN
df = df.where(
    (df["city"] == "MN, Minneapolis")
    & (df["building_type"] == "Single-Family Detached" ), axis=0
).dropna()

In [34]:
df['building_id'].unique()

array(['1641', '6250', '9572', '13087', '13786', '22043', '27407',
       '30507', '31874', '32331', '35520', '38425', '40952', '45931',
       '47534', '57625', '65224', '67464', '72940', '81997', '82186',
       '82862', '83178', '86765', '90349', '92004', '94083', '98308',
       '98992', '100745', '101421', '103386', '103928', '105194',
       '108104', '111805', '125109', '125389', '133093', '138909',
       '153329', '153974', '157454', '159647', '161029', '161714',
       '165633', '167877', '171825', '173238', '179657', '181617',
       '181832', '182925', '190720', '191361', '198065', '200457',
       '205050', '205881', '209362', '213254', '213931', '223019',
       '227783', '231096', '234796', '235665', '235990', '236632',
       '237987', '240687', '241727', '245316', '252437', '253128',
       '253632', '253964', '254216', '254612', '257267', '261019',
       '268305', '269531', '272747', '272790', '274938', '276276',
       '281120', '283116', '284960', '286536', '292265

In [ ]:
# Standard baseline R-values for uninsulated (unfinished) building elements
UNFINISHED_R_VALUES = {
    "floor_r": 2.0,  # 4" concrete slab + ground contact + air film
    "foundation_wall_r": 2.0,  # 8" poured concrete wall + air films
    "ceiling_r": 2.5,  # 5/8" drywall/joists + still air film
    "rim_joist_r": 2.5,  # 1.5" wood rim board + sheathing + air films
    "roof_r": 3.0,  # 5/8" OSB decking + asphalt shingles + air films
}

def parse_r_value(val: str, default_r: float, col_name: str = "not specified") -> float:
    """Extracts the numeric float directly following 'r-' or 'R-'.
    Defaults to default_r if 'r-<number>' isn't present or val is missing.
    """
    if pd.isna(val):
        return default_r

    # Looks specifically for 'r-' or 'R-' followed by digits/decimals
    match = re.search(r'r-(\d+(?:\.\d+)?)', str(val), flags=re.IGNORECASE)

    return float(match.group(1)) if match else default_r


def calculate_unified_r(
    df: pd.DataFrame, area_weights: dict = None
) -> pd.DataFrame:
    """Parses string R-values to floats and computes a unified parallel envelope R-value."""
    df_out = df.copy()
    r_cols = list(UNFINISHED_R_VALUES.keys())
    num_cols = []

    # Parse columns into numeric floats
    for col in r_cols:
        num_col = f"{col}_float"
        default = UNFINISHED_R_VALUES[col]
        df_out[num_col] = df_out[col].apply(
            lambda x: parse_r_value(x, default, col_name=col)
        )
        num_cols.append(num_col)

    # Parallel Heat Flow Calculation: U_total = sum(U_i) -> R_unified = 1 / U_avg
    u_values = 1.0 / df_out[num_cols].values

    if area_weights:
        # Area-weighted harmonic mean: R_unified = sum(A_i) / sum(A_i / R_i)
        weights = np.array([area_weights[col] for col in r_cols])
        df_out["unified_r_value"] = weights.sum() / (u_values @ weights)
    else:
        # Unweighted harmonic mean across the 5 assemblies
        df_out["unified_r_value"] = len(r_cols) / u_values.sum(axis=1)

    return df_out


df['ceiling_r'].unique()


df = calculate_unified_r(df)

In [19]:
df['building_type'].unique()

array(['Single-Family Detached', 'Single-Family Attached'], dtype=object)

In [84]:
def find_closest_row(df, heating_fuel, heating_setpoint, unified_r, area):
    numeric_columns = ["heating_setpoint", "unified_r_value", "area"]
    numeric_values = df[numeric_columns].copy()
    numeric_values["heating_setpoint"] = pd.to_numeric(
        numeric_values["heating_setpoint"]
        .astype("string")
        .str.extract(r"([-+]?\d+(?:\.\d+)?)", expand=False),
        errors="coerce",
    )
    numeric_values[["unified_r_value", "area"]] = numeric_values[
        ["unified_r_value", "area"]
    ].apply(pd.to_numeric, errors="coerce")

    candidates = df[df["heating_fuel"] == heating_fuel]
    if candidates.empty:
        raise ValueError(f"No rows found for heating_fuel={heating_fuel!r}")

    candidate_values = numeric_values.loc[candidates.index].dropna()
    if candidate_values.empty:
        raise ValueError(f"No numeric rows found for heating_fuel={heating_fuel!r}")

    setpoint = pd.to_numeric(
        pd.Series([heating_setpoint], dtype="string").str.extract(
            r"([-+]?\d+(?:\.\d+)?)", expand=False
        ).iloc[0],
        errors="coerce",
    )
    target = pd.Series([setpoint, unified_r, area], index=numeric_columns, dtype=float)
    scales = numeric_values.std().replace(0, 1)
    distances = (((candidate_values - target) / scales) ** 2).sum(axis=1)
    return df.loc[distances.idxmin()]


In [88]:
find_closest_row(df, heating_fuel="Natural Gas", heating_setpoint=70, unified_r=15, area=2000)

bldg_id                                         274938.0
city                                     MN, Minneapolis
area                                              3310.0
heating_fuel                                 Natural Gas
heating_setpoint                                     68F
ceiling_r                                           R-49
floor_r                                      Uninsulated
foundation_wall_r                    Wall R-15, Exterior
rim_joist_r                               R-15, Exterior
roof_r                           Unfinished, Uninsulated
heat_pump_backup_primary_size                        0.0
heating_system_primary_size                        90.93
heating_system_secondary_size                        0.0
floor_r_float                                        2.0
foundation_wall_r_float                             15.0
ceiling_r_float                                     49.0
rim_joist_r_float                                   15.0
roof_r_float                   

In [ ]:
# os.path.isfile(get_parquet_file_path("MN", '274938'))


True